# Spindle — Streaming & Anomaly Injection

Spindle can stream synthetic events row-by-row with realistic arrival patterns — ideal for feeding **Microsoft Fabric Eventstream**, Azure Event Hubs, or Kafka topics.

**Features**
- Poisson inter-arrival times (statistically realistic pacing)
- Token-bucket rate limiting with configurable burst windows
- Out-of-order event injection (simulates late-arriving data)
- Three anomaly types: point, contextual, collective
- Sinks: `ConsoleSink`, `FileSink` (built-in); `EventHubSink`, `KafkaSink` (`pip install sqllocks-spindle[streaming]`)

**Sinks**
| Sink | Install | Use case |
|---|---|---|
| `ConsoleSink` | built-in | Development / debugging |
| `FileSink` | built-in | Local JSONL files |
| `EventHubSink` | `[streaming]` | Fabric Eventstream, Azure Event Hubs |
| `KafkaSink` | `[streaming]` | Apache Kafka |


In [ ]:
%pip install sqllocks-spindle --quiet
# For Event Hub / Kafka support:
# %pip install sqllocks-spindle[streaming] --quiet

## Basic streaming — console sink

`realtime=False` emits events as fast as possible (best for Fabric notebooks).
Set `realtime=True` to enable Poisson inter-arrivals + token-bucket rate limiting.

In [ ]:
from sqllocks_spindle import RetailDomain
from sqllocks_spindle.streaming import SpindleStreamer, StreamConfig, ConsoleSink

result = SpindleStreamer(
    domain=RetailDomain(),
    sink=ConsoleSink(max_print=5),          # print first 5 events, then suppress
    config=StreamConfig(max_events=20, realtime=False),
    scale="fabric_demo",
    seed=42,
).stream("order")

print(result)

## File sink — JSONL output

Write events to a JSONL file for inspection or downstream processing.

In [ ]:
import tempfile, os, json
from sqllocks_spindle.streaming import FileSink

with tempfile.TemporaryDirectory() as tmpdir:
    outfile = os.path.join(tmpdir, "orders.jsonl")

    result = SpindleStreamer(
        domain=RetailDomain(),
        sink=FileSink(outfile, mode="w"),
        config=StreamConfig(max_events=50, realtime=False),
        scale="fabric_demo",
        seed=42,
    ).stream("order")

    print(result)

    # Inspect a few lines
    with open(outfile) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(json.loads(line))

## Anomaly injection

Inject synthetic anomalies into the stream. All anomalous events are tagged with `_spindle_is_anomaly=True` and `_spindle_anomaly_type`.

In [ ]:
from sqllocks_spindle.streaming import AnomalyRegistry, PointAnomaly, ContextualAnomaly

registry = AnomalyRegistry([
    # 5% of events: order_total spiked to 10x normal range
    PointAnomaly(column="order_total", fraction=0.05, scale_factor=10.0),

    # Cancelled orders sometimes still have high totals (contextual)
    ContextualAnomaly(
        column="order_total",
        condition_column="status",
        condition_values=["cancelled"],
        replacement_range=(500.0, 2000.0),
        fraction=0.30,
    ),
])

with tempfile.TemporaryDirectory() as tmpdir:
    outfile = os.path.join(tmpdir, "orders_with_anomalies.jsonl")

    result = SpindleStreamer(
        domain=RetailDomain(),
        sink=FileSink(outfile, mode="w"),
        config=StreamConfig(max_events=200, realtime=False),
        anomaly_registry=registry,
        scale="fabric_demo",
        seed=42,
    ).stream("order")

    print(result)
    print(f"Anomalies injected: {result.anomaly_count} / {result.events_sent} ({result.anomaly_count/result.events_sent:.1%})")

## Burst windows

Simulate traffic spikes — e.g. flash sales, Black Friday, end-of-month billing.

`BurstWindow(start_offset_s, duration_s, multiplier)` — after `start_offset_s` seconds, emit at `multiplier`× the base rate for `duration_s` seconds.

In [ ]:
from sqllocks_spindle.streaming import BurstWindow

config = StreamConfig(
    rate=50.0,          # 50 events/sec baseline
    realtime=True,      # enable rate limiting
    max_events=150,
    burst_windows=[
        BurstWindow(start_offset_s=1.0, duration_s=2.0, multiplier=5.0),  # 5× spike at t=1s
    ],
)

print("BurstWindow config ready.")
print(f"  Baseline rate: {config.rate} events/sec")
print(f"  Burst at t=1s: 5× for 2 seconds ({config.rate * 5:.0f} events/sec)")
print()
print("Run SpindleStreamer with this config to observe burst behavior.")
print("(Omitting live run here to avoid blocking the notebook for 3+ seconds.)")

## Event Hub sink — Fabric Eventstream

Requires `pip install sqllocks-spindle[streaming]` and an Azure Event Hubs connection string.

In Fabric, create an **Eventstream** → add a **Custom App** source → copy the connection string.

In [ ]:
# Uncomment and fill in your connection string to run

# from sqllocks_spindle.streaming import EventHubSink
#
# CONNECTION_STRING = "Endpoint=sb://..."  # from Fabric Eventstream Custom App source
# EVENTHUB_NAME    = "spindle-retail"
#
# result = SpindleStreamer(
#     domain=RetailDomain(),
#     sink=EventHubSink(CONNECTION_STRING, eventhub_name=EVENTHUB_NAME),
#     config=StreamConfig(rate=100.0, realtime=True, max_events=1000),
#     scale="fabric_demo",
#     seed=42,
# ).stream("order")
#
# print(result)

print("Fill in CONNECTION_STRING and EVENTHUB_NAME above, then uncomment to run.")

## Stream multiple tables

Run a streamer per table to simulate concurrent event streams.

In [ ]:
from sqllocks_spindle import Spindle

# Pre-generate tables once, then stream each independently
spindle = Spindle()
batch = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

for table_name in ["order", "order_line", "return"]:
    with tempfile.TemporaryDirectory() as tmpdir:
        r = SpindleStreamer(
            tables=batch.tables,
            sink=FileSink(os.path.join(tmpdir, f"{table_name}.jsonl"), mode="w"),
            config=StreamConfig(max_events=30, realtime=False),
        ).stream(table_name)
        print(f"  {table_name:<12} {r}")